### Introduction to Machine Learning
Jesús Urtasun Elizari: MRC LMS 2026

**Chapter 4: Generative AI. Exercise 1**

**Markov chains: states are vectors, transitions are matrices**

The weather tomorrow is random, but it depends on the weather today: after a sunny day, another sunny day is more likely.
We write the weather as a vector of probabilities and the changes from one day to the next as a matrix, first with two states (sunny, rainy) and then with three (sunny, cloudy, rainy).

**The maths**

The state on day $t$ is a vector with the probability of each weather, for example $p_t = (P(\text{sunny}), P(\text{rainy}))$. Its entries add up to 1.

The transition matrix $P$ has one row for today's weather and one column for tomorrow's: $P_{ij}$ is the probability of going from weather $i$ to weather $j$. Each row adds up to 1. Tomorrow's state is today's state times the matrix:

$$p_{t+1} = p_t \, P \qquad \begin{pmatrix} p_{\text{sunny}} & p_{\text{rainy}} \end{pmatrix}_{t+1} = \begin{pmatrix} p_{\text{sunny}} & p_{\text{rainy}} \end{pmatrix}_{t} \begin{pmatrix} P_{\text{sunny} \to \text{sunny}} & P_{\text{sunny} \to \text{rainy}} \\ P_{\text{rainy} \to \text{sunny}} & P_{\text{rainy} \to \text{rainy}} \end{pmatrix}$$

This is the Markov property: tomorrow depends only on today, not on the days before (memoryless). After many days the state stops changing and reaches the stationary state $\pi$:

$$\pi = \pi \, P$$

In [ ]:
# Import libraries
import numpy as np
import matplotlib.pyplot as plt

# Check installation
print("All libraries imported successfully")
print(np.__version__)

In [ ]:
# Two states: sunny and rainy
states = ['sunny', 'rainy']

# Transition matrix: row = today, column = tomorrow
P = np.array([
    [0.8, 0.2],   # today sunny: 80% sunny tomorrow, 20% rainy
    [0.4, 0.6]])  # today rainy: 40% sunny tomorrow, 60% rainy
print("P:\n", P)
print("Shape:", P.shape)
print("Rows add up to:", P.sum(axis = 1))

In [ ]:
# Today is sunny for sure
p = np.array([1.0, 0.0])
print("Day 0:", p)

# Tomorrow: vector times matrix
p = p @ P
print("Day 1:", p)

# The day after
p = p @ P
print("Day 2:", p)

In [ ]:
# Follow the state for 15 days
p = np.array([1.0, 0.0])
history = [p]
for day in range(15):
    p = p @ P
    history.append(p)
history = np.array(history)
print("Day 15:", np.round(p, 3))

plt.figure(figsize = (8, 5))
plt.plot(history[:, 0], marker = 'o', color = 'orange', label = 'sunny')
plt.plot(history[:, 1], marker = 'o', color = 'blue', label = 'rainy')
plt.xlabel('day')
plt.ylabel('probability')
plt.title('Two states: the probabilities settle down')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Check with numpy implementation
# The stationary state is the eigenvector of P transposed with eigenvalue 1
eigenvalues, eigenvectors = np.linalg.eig(P.T)
pi = np.real(eigenvectors[:, np.argmax(np.real(eigenvalues))])
pi = pi / np.sum(pi)
print("Stationary state (eigenvector):", np.round(pi, 3))
print("After 15 days:                 ", np.round(history[-1], 3))

In [ ]:
# Three states: sunny, cloudy, rainy
states = ['sunny', 'cloudy', 'rainy']

P3 = np.array([
    [0.7, 0.2, 0.1],   # today sunny
    [0.3, 0.4, 0.3],   # today cloudy
    [0.2, 0.3, 0.5]])  # today rainy
print("P3:\n", P3)
print("Shape:", P3.shape)
print("Rows add up to:", P3.sum(axis = 1))

In [ ]:
# Today is rainy for sure, follow the state for 15 days
p = np.array([0.0, 0.0, 1.0])
history = [p]
for day in range(15):
    p = p @ P3
    history.append(p)
history = np.array(history)
print("Day 1: ", np.round(history[1], 3))
print("Day 15:", np.round(history[-1], 3))

plt.figure(figsize = (8, 5))
plt.plot(history[:, 0], marker = 'o', color = 'orange', label = 'sunny')
plt.plot(history[:, 1], marker = 'o', color = 'gray', label = 'cloudy')
plt.plot(history[:, 2], marker = 'o', color = 'blue', label = 'rainy')
plt.xlabel('day')
plt.ylabel('probability')
plt.title('Three states: the probabilities settle down')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Simulate the weather day by day: pick tomorrow at random using the row of today
np.random.seed(0)
today = 2   # rainy
weather = [states[today]]
for day in range(14):
    today = np.random.choice(3, p = P3[today])
    weather.append(states[today])
print("Two weeks of weather:\n", weather)

In [ ]:
# Check: over many days, the fraction of each weather matches the stationary state
today = 2
counts = np.zeros(3)
for day in range(10000):
    today = np.random.choice(3, p = P3[today])
    counts[today] += 1
print("Fraction over 10000 days:", np.round(counts / 10000, 3))
print("Stationary state:        ", np.round(history[-1], 3))